In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt

team_data = pd.read_csv("data/team_data.csv")

team_data.info()

# purely scalar/numerical data
columns = [
    "comps_attended",
    "sigs_attended",
    "trueskill",
    "trueskill_ranking",
    "ccwm",
    "opr",
    "dpr",
    "ap_per_match",
    "awp_per_match",
    "wp_per_match",
    "drive_score",
    "auto_score"
]

# Choose only numerical for K-Means
VARIABLES = ["comps_attended", "trueskill", "ccwm", "drive_score", "awp_per_match", "sigs_attended", "worlds_qual_numerical"]

cluster_range = range(3, 6)
variable_range = range(3, 6)

def numerical_worlds_qual(wq: bool):
    if wq:
        return 1
    else:
        return 0

team_data["worlds_qual_numerical"] = team_data["worlds_qual"].apply(numerical_worlds_qual)

team_data


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from itertools import combinations
import seaborn as sns
import os

combinations_list = []
for r in variable_range: combinations_list.extend(combinations(VARIABLES, r))


def run_cluster(vars_used: list[str], clust_count: int):
    modified_dataframe = team_data[vars_used]
    kmeans =  KMeans(n_clusters=clust_count, random_state=0, n_init="auto").fit(modified_dataframe)
    modified_dataframe['cluster'] = kmeans.labels_
    return kmeans, vars_used, modified_dataframe

def correlation_heatmap(df: pd.DataFrame, variables: list[str]):
    plt.figure(figsize=(10, 8))
    correlation_matrix = df[variables].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Heatmap of Variables')
    return plt


def pca_loadings_heatmap(pca_object: PCA, variables_list: list[str]):
    loadings = pca_object.components_

    loadings_df = pd.DataFrame(loadings,
                               columns=variables_list, 
                               index=[f'PC{i+1}' for i in range(loadings.shape[0])])

    plt.figure(figsize=(10, max(6, len(variables_list) * 0.7)))
    sns.heatmap(loadings_df, annot=True, cmap='vlag', fmt=".2f", linewidths=.5)
    plt.title('PCA Component Loadings Heatmap')
    plt.xlabel('Original Variables')
    plt.ylabel('Principal Components')
    plt.yticks(rotation=0)
    plt.tight_layout() 
    return plt


def pca_visualization_2d(clustered_df: pd.DataFrame, variables: list[str], clust_count: int):
    if len(variables) > 2:
        pca = PCA(n_components=2)
        principal_components = pca.fit_transform(clustered_df[variables])
        pca_df = pd.DataFrame(data=principal_components, columns=['principal_component_1', 'principal_component_2'])
        pca_df['cluster'] = clustered_df['cluster']

        plt.figure(figsize=(12, 10))
        sns.scatterplot(x='principal_component_1', y='principal_component_2', hue='cluster', data=pca_df, palette='viridis', s=100)
        plt.title(f'2d K-Means ({clust_count}), {variables}')
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.legend(title='Cluster')
        plt.grid(True)

        return pca, plt
    
def pca_visualization_3d(clustered_df: pd.DataFrame, variables: list[str], clust_count: int):
    if len(variables) > 2:
        pca = PCA(n_components=3)
        principal_components = pca.fit_transform(clustered_df[variables].values)

        pca_df = pd.DataFrame(data=principal_components,
                              columns=['principal_component_1', 'principal_component_2', 'principal_component_3'])
        pca_df['cluster'] = clustered_df['cluster'].reset_index(drop=True)

        fig = plt.figure(figsize=(14, 12))
        ax = fig.add_subplot(111, projection='3d')
        unique_clusters = sorted(pca_df['cluster'].unique())
        cmap = plt.cm.get_cmap('viridis', len(unique_clusters))


        for i, cluster_label in enumerate(unique_clusters):
            cluster_data = pca_df[pca_df['cluster'] == cluster_label]
            ax.scatter(cluster_data['principal_component_1'],
                       cluster_data['principal_component_2'],
                       cluster_data['principal_component_3'],
                       c=[cmap(i)], 
                       label=f'Cluster {cluster_label}',
                       s=40, 
                       alpha=0.2) 

        ax.set_title(f'3d K-Means ({clust_count}), {variables}')
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3') 
        ax.legend(title='Cluster')
        ax.grid(True)


        return pca, plt



results = {}
for n_clusters in cluster_range:
    for combo in combinations_list:
        model, variables, data = run_cluster(list(combo), n_clusters)


        stringified_vars = "_".join(combo)
        folder_name = f"cluster_results/{stringified_vars}-{n_clusters}-clusters"
        os.makedirs(folder_name, exist_ok=True)
        os.makedirs(folder_name + "/2d", exist_ok=True)
        os.makedirs(folder_name + "/3d", exist_ok=True)


        pca, pca_graph = pca_visualization_3d(data, variables, n_clusters)
        pca_graph.savefig(os.path.join(folder_name, "3d/pca.png"), bbox_inches='tight')

        pca_heatmap_graph = pca_loadings_heatmap(pca, variables)
        pca_heatmap_graph.savefig(os.path.join(folder_name, "3d/loadings.png"), bbox_inches='tight')


        pca_2d, pca_2d_graph = pca_visualization_2d(data, variables, n_clusters)
        pca_2d_graph.savefig(os.path.join(folder_name, "2d/pca.png"), bbox_inches='tight')


        pca_2d_heatmap_graph = pca_loadings_heatmap(pca_2d, variables)
        pca_2d_heatmap_graph.savefig(os.path.join(folder_name, "2d/loadings.png"), bbox_inches='tight')

    
        corr_heatmap = correlation_heatmap(data, variables)
        corr_heatmap.savefig(os.path.join(folder_name, "correlation_heatmap.png"), bbox_inches='tight')

        


